In [ ]:
!pip install bitsandbytes==0.46.1 --no-cache-dir

In [2]:
import bitsandbytes as bnb
print("BitsAndBytes version:", bnb.__version__)

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


BitsAndBytes version: 0.46.1
CUDA available: True
GPU: Tesla T4


In [4]:
"""
Assignment: Deploying Mistral-7B-Instruct on Resource-Constrained GPU
Using 8-bit quantization (Python 3.12 compatible)
GPU: Tesla T4

Note:
4-bit quantization is not compatible with Python 3.12 in this environment.
Therefore, 8-bit quantization is used as a stable alternative.
"""

# ==========================================
# 1. Install Required Libraries (Run Once)
# ==========================================
# !pip install -U transformers accelerate bitsandbytes

# ==========================================
# 2. Import Required Libraries
# ==========================================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# ==========================================
# 3. Check GPU Availability
# ==========================================

if not torch.cuda.is_available():
    raise EnvironmentError("GPU not available. Please enable GPU.")

print("GPU is available.")
print("GPU Name:", torch.cuda.get_device_name(0))

# ==========================================
# 4. Define Model ID
# ==========================================

model_id = "mistralai/Mistral-7B-Instruct-v0.2"

# ==========================================
# 5. Configure 8-bit Quantization
# ==========================================

# 8-bit quantization is stable with Python 3.12
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# ==========================================
# 6. Load Tokenizer
# ==========================================

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

# ==========================================
# 7. Load Model with Quantization
# ==========================================

print("Loading model in 8-bit mode...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)

model.eval()
print("Model loaded successfully.")

# ==========================================
# 8. Define Text Generation Function
# ==========================================

def generate_response(prompt, max_new_tokens=150):
    """
    Generates response using Mistral instruction format.
    max_new_tokens kept at 150 as required.
    """
    
    # Mistral requires instruction formatting
    formatted_prompt = f"<s>[INST] {prompt} [/INST]"
    
    # Tokenize input
    inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)
    
    # Disable gradients for inference
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True
        )
    
    # Decode output
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    return response

# ==========================================
# 9. Zero-Shot Prompt
# ==========================================

print("\n===== ZERO-SHOT PROMPT =====\n")

zero_shot_prompt = "Explain Graph Neural Networks in one paragraph."

zero_shot_output = generate_response(zero_shot_prompt)

print(zero_shot_output)

# ==========================================
# 10. Few-Shot Prompt
# ==========================================

print("\n===== FEW-SHOT PROMPT =====\n")

few_shot_prompt = """
Translate the following English sentences to French:

English: I love machine learning.
French: J'aime l'apprentissage automatique.

English: The weather is beautiful today.
French: Le temps est magnifique aujourd'hui.

English: Artificial intelligence is transforming the world.
French:
"""

few_shot_output = generate_response(few_shot_prompt)

print(few_shot_output)

# ==========================================
# 11. Display GPU Memory Usage
# ==========================================

print("\nGPU Memory Used (GB):", torch.cuda.memory_allocated() / 1e9)


GPU is available.
GPU Name: Tesla T4
Loading tokenizer...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


Model loaded successfully.

===== ZERO-SHOT PROMPT =====



/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:185: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[INST] Explain Graph Neural Networks in one paragraph. [/INST] Graph Neural Networks (GNNs) are a type of artificial intelligence model specifically designed to learn and make predictions or discover hidden patterns in data represented as graphs. A graph is a set of nodes (vertices) and edges (connections) that represent relationships between the nodes. In a GNN, each node is assigned a feature vector and the model learns to update these features based on the information of its neighboring nodes and the edges connecting them. The updates are typically performed in a message-passing or propagation step, followed by an aggregation step where the node integrates the received messages. GNNs can learn more complex and nuanced representations of data compared to traditional neural networks, making them particularly useful for tasks

===== FEW-SHOT PROMPT =====

[INST] 
Translate the following English sentences to French:

English: I love machine learning.
French: J'aime l'apprentissage autom